<a href="https://colab.research.google.com/github/engMohamedAbdAlslam/DRP_segmentation/blob/copilot%2Fdevelop-preprocessing-pipeline/notebooks/06_gradcam_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 06 — Grad-CAM + Vessel Segmentation Fusion Pipeline
**Goal:** Combine Grad-CAM heatmap (from EfficientNet-B4 grading model) with vessel segmentation mask (from U-Net) to produce a single unified clinical overlay per retinal image.

**Inputs:**
- Preprocessed fundus image (`.npz` from NB01 or raw image)
- Trained DR grading model: `best_efficientnet_b4.pth` (from NB04)
- Trained vessel segmentation model: `best_vessel_unet.pth` (from NB02)

**Output:** Fused overlay image showing:
- Grad-CAM heatmap (disease-relevant regions)
- Vessel segmentation mask (green overlay)
- Combined overlay for clinician review

> Run on **Google Colab** with T4 GPU.

## 1. Colab Repo Setup

In [ ]:
import os
from pathlib import Path

repo_path = Path('/content/DRP_segmentation')
if not repo_path.exists():
    !git clone https://github.com/engMohamedAbdAlslam/DRP_segmentation.git /content/DRP_segmentation
%cd /content/DRP_segmentation
!git checkout copilot/develop-preprocessing-pipeline
!git pull origin copilot/develop-preprocessing-pipeline
print('Repo ready at', Path.cwd())

## 2. Install Dependencies

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', '-q', 'install',
                'timm', 'segmentation-models-pytorch', 'albumentations',
                'opencv-python-headless', 'matplotlib', 'numpy', 'torch', 'torchvision'],
               check=True)
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 3. Imports & Config

In [ ]:
import cv2
import numpy as np
import torch
import torch.nn as nn
import timm
import segmentation_models_pytorch as smp
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from google.colab import drive

DEVICE        = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
NUM_CLASSES   = 5
IMAGE_SIZE    = 380
VESSEL_SIZE   = 512
VES_THRESHOLD = 0.5

GRADE_NAMES = {
    0: 'No DR',
    1: 'Mild DR',
    2: 'Moderate DR',
    3: 'Severe DR',
    4: 'Proliferative DR'
}

GRADE_COLORS = {
    0: (0, 200, 0),
    1: (255, 200, 0),
    2: (255, 140, 0),
    3: (255, 70, 0),
    4: (220, 0, 0)
}

print(f'Device: {DEVICE}')

## 4. Mount Drive & Load Models

In [ ]:
drive.mount('/content/drive')

GRADING_MODEL_PATH = Path('/content/drive/MyDrive/DRP_models/best_efficientnet_b4.pth')
VESSEL_MODEL_PATH  = Path('/content/drive/MyDrive/DRP_models/best_vessel_unet.pth')
TEST_IMAGES_DIR    = Path('/content/drive/MyDrive/DRP_processed/aptos2019/test')
OUTPUT_DIR         = Path('/content/drive/MyDrive/DRP_outputs/nb06_fused')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('Drive mounted.')
print(f'Grading model exists: {GRADING_MODEL_PATH.exists()}')
print(f'Vessel model exists:  {VESSEL_MODEL_PATH.exists()}')
print(f'Test images found:    {len(list(TEST_IMAGES_DIR.rglob("*.npz")))} files')

## 5. Build & Load Grading Model (EfficientNet-B4)

In [ ]:
grading_model = timm.create_model('efficientnet_b4', pretrained=False, num_classes=NUM_CLASSES)
grading_model.load_state_dict(torch.load(GRADING_MODEL_PATH, map_location=DEVICE))
grading_model = grading_model.to(DEVICE)
grading_model.eval()
print('Grading model loaded.')

## 6. Build & Load Vessel Segmentation Model (U-Net + EfficientNet-B3)

In [ ]:
vessel_model = smp.Unet(
    encoder_name='efficientnet-b3',
    encoder_weights=None,
    in_channels=3,
    classes=1
)
vessel_model.load_state_dict(torch.load(VESSEL_MODEL_PATH, map_location=DEVICE))
vessel_model = vessel_model.to(DEVICE)
vessel_model.eval()
print('Vessel segmentation model loaded.')

## 7. Grad-CAM Implementation

In [ ]:
class GradCAM:
    """
    Grad-CAM for EfficientNet-B4 from timm.
    Target layer: model.blocks[-1] (last convolutional block).
    """
    def __init__(self, model):
        self.model = model
        self.gradients = None
        self.activations = None
        self._register_hooks()

    def _register_hooks(self):
        target_layer = self.model.blocks[-1]

        def forward_hook(module, input, output):
            self.activations = output.detach()

        def backward_hook(module, grad_input, grad_output):
            self.gradients = grad_output[0].detach()

        target_layer.register_forward_hook(forward_hook)
        target_layer.register_full_backward_hook(backward_hook)

    def generate(self, input_tensor, target_class=None):
        """
        Generate Grad-CAM heatmap.
        Returns normalized heatmap [0, 1] as numpy array.
        """
        self.model.eval()
        input_tensor = input_tensor.to(DEVICE).requires_grad_(True)

        output = self.model(input_tensor)
        probs  = torch.softmax(output, dim=1)

        if target_class is None:
            target_class = output.argmax(dim=1).item()
        predicted_class = output.argmax(dim=1).item()
        confidence = probs[0, predicted_class].item()

        self.model.zero_grad()
        output[0, target_class].backward()

        weights = self.gradients.mean(dim=[2, 3], keepdim=True)
        cam = (weights * self.activations).sum(dim=1, keepdim=True)
        cam = torch.relu(cam)
        cam = torch.nn.functional.interpolate(
            cam, size=(IMAGE_SIZE, IMAGE_SIZE), mode='bilinear', align_corners=False
        )
        cam = cam.squeeze().cpu().numpy()
        cam_min, cam_max = cam.min(), cam.max()
        if cam_max - cam_min > 1e-8:
            cam = (cam - cam_min) / (cam_max - cam_min)
        else:
            cam = np.zeros_like(cam)

        return cam, predicted_class, confidence, probs[0].detach().cpu().numpy()


gradcam = GradCAM(grading_model)
print('Grad-CAM engine ready.')

## 8. Preprocessing Helpers

In [ ]:
from torchvision import transforms

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]
VESSEL_MEAN   = [0.6129, 0.2364, 0.1281]
VESSEL_STD    = [0.3016, 0.1446, 0.0837]

grading_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

vessel_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=VESSEL_MEAN, std=VESSEL_STD)
])


def preprocess_for_grading(image_bgr):
    img_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    img_resized = cv2.resize(img_rgb, (IMAGE_SIZE, IMAGE_SIZE), interpolation=cv2.INTER_CUBIC)
    tensor = grading_transform(img_resized).unsqueeze(0)
    return tensor, img_resized


def preprocess_for_vessel(image_bgr):
    green = image_bgr[:, :, 1]
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    enhanced = clahe.apply(green)
    img_3ch = np.stack([enhanced] * 3, axis=-1)
    img_resized = cv2.resize(img_3ch, (VESSEL_SIZE, VESSEL_SIZE), interpolation=cv2.INTER_CUBIC)
    tensor = vessel_transform(img_resized).unsqueeze(0)
    return tensor, img_resized


print('Preprocessing helpers ready.')

## 9. Fusion Pipeline

In [ ]:
def run_vessel_segmentation(image_bgr):
    tensor, _ = preprocess_for_vessel(image_bgr)
    with torch.no_grad():
        output = vessel_model(tensor.to(DEVICE))
        prob_map = torch.sigmoid(output).squeeze().cpu().numpy()
    binary_mask = (prob_map > VES_THRESHOLD).astype(np.uint8)
    binary_mask_resized = cv2.resize(
        binary_mask, (IMAGE_SIZE, IMAGE_SIZE), interpolation=cv2.INTER_NEAREST
    )
    return binary_mask_resized, prob_map


def create_fused_overlay(original_rgb, gradcam_heatmap, vessel_mask, predicted_class, confidence):
    H, W = original_rgb.shape[:2]
    base = original_rgb.copy().astype(np.float32)

    # Grad-CAM layer
    heatmap_uint8 = (gradcam_heatmap * 255).astype(np.uint8)
    heatmap_colored = cv2.applyColorMap(heatmap_uint8, cv2.COLORMAP_JET)
    heatmap_rgb = cv2.cvtColor(heatmap_colored, cv2.COLOR_BGR2RGB).astype(np.float32)
    fused = cv2.addWeighted(base, 0.55, heatmap_rgb, 0.45, 0)

    # Vessel mask layer
    vessel_layer = np.zeros_like(base)
    vessel_layer[vessel_mask == 1] = [0, 220, 80]
    fused = cv2.addWeighted(fused, 0.70, vessel_layer, 0.30, 0)

    fused = np.clip(fused, 0, 255).astype(np.uint8)

    # Grade label
    grade_color_rgb = GRADE_COLORS[predicted_class]
    grade_color_bgr = (grade_color_rgb[2], grade_color_rgb[1], grade_color_rgb[0])
    label = f"{GRADE_NAMES[predicted_class]} ({confidence*100:.1f}%)"
    cv2.rectangle(fused, (0, H - 36), (W, H), (0, 0, 0), -1)
    cv2.putText(fused, label, (8, H - 10),
                cv2.FONT_HERSHEY_SIMPLEX, 0.65, grade_color_bgr, 2, cv2.LINE_AA)

    return fused


def process_image(image_path_or_array, save_path=None, show=True):
    """
    Full fusion pipeline for a single image.
    Accepts file path (str/Path) or a BGR numpy array.
    """
    if isinstance(image_path_or_array, (str, Path)):
        p = Path(image_path_or_array)
        if p.suffix == '.npz':
            data = np.load(p, allow_pickle=True)
            img_rgb = data['image'].astype(np.uint8)
            img_bgr = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR)
        else:
            img_bgr = cv2.imread(str(p))
    else:
        img_bgr = image_path_or_array

    # Grading + Grad-CAM
    grade_tensor, img_rgb_380 = preprocess_for_grading(img_bgr)
    heatmap, pred_class, confidence, all_probs = gradcam.generate(grade_tensor)

    # Vessel Segmentation
    vessel_mask, vessel_prob = run_vessel_segmentation(img_bgr)

    # Fusion
    fused = create_fused_overlay(img_rgb_380, heatmap, vessel_mask, pred_class, confidence)

    if show:
        fig, axes = plt.subplots(1, 4, figsize=(20, 5))

        axes[0].imshow(img_rgb_380)
        axes[0].set_title('Original')

        heatmap_colored = cv2.applyColorMap((heatmap * 255).astype(np.uint8), cv2.COLORMAP_JET)
        heatmap_rgb_disp = cv2.cvtColor(heatmap_colored, cv2.COLOR_BGR2RGB)
        overlay = cv2.addWeighted(img_rgb_380, 0.55, heatmap_rgb_disp, 0.45, 0)
        axes[1].imshow(overlay)
        axes[1].set_title(f'Grad-CAM\n{GRADE_NAMES[pred_class]} ({confidence*100:.1f}%)')

        axes[2].imshow(vessel_mask, cmap='Greens')
        axes[2].set_title('Vessel Segmentation')

        axes[3].imshow(cv2.cvtColor(fused, cv2.COLOR_BGR2RGB))
        axes[3].set_title('Fused Clinical Overlay')

        for ax in axes:
            ax.axis('off')

        legend_elements = [
            mpatches.Patch(color='lime',  label='Vessel segments'),
            mpatches.Patch(color='red',   label='High-risk regions (Grad-CAM)'),
            mpatches.Patch(color='blue',  label='Low-risk regions (Grad-CAM)'),
        ]
        fig.legend(handles=legend_elements, loc='lower center', ncol=3, fontsize=10)
        plt.tight_layout(rect=[0, 0.07, 1, 1])

        if save_path:
            plt.savefig(save_path, dpi=150, bbox_inches='tight')
            print(f'Saved: {save_path}')
        plt.show()

    return {
        'original_rgb': img_rgb_380,
        'gradcam':      heatmap,
        'vessel_mask':  vessel_mask,
        'fused':        fused,
        'pred_class':   pred_class,
        'confidence':   confidence,
        'all_probs':    all_probs,
        'grade_name':   GRADE_NAMES[pred_class]
    }


print('Fusion pipeline ready.')

## 10. Run on Sample Images

In [ ]:
test_files = sorted(TEST_IMAGES_DIR.rglob('*.npz'))
print(f'Found {len(test_files)} test images')

grade_samples = {}
for f in test_files:
    grade = int(f.stem.split('_grade')[-1])
    if grade not in grade_samples:
        grade_samples[grade] = f

print(f'Running fusion on {len(grade_samples)} images (one per grade)...')
for grade, img_path in sorted(grade_samples.items()):
    print(f'\n--- Grade {grade}: {GRADE_NAMES[grade]} ---')
    result = process_image(
        img_path,
        save_path=OUTPUT_DIR / f'fused_grade{grade}.png',
        show=True
    )
    print(f'Predicted: {result["grade_name"]} ({result["confidence"]*100:.1f}%)')
    print(f'All probs: {dict(zip(GRADE_NAMES.values(), result["all_probs"].round(3)))}')

## 11. Batch Processing

In [ ]:
from tqdm import tqdm

def batch_process(image_paths, output_dir, max_images=50):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    results = []

    for img_path in tqdm(image_paths[:max_images], desc='Processing images'):
        save_path = output_dir / f"{Path(img_path).stem}_fused.png"
        try:
            result = process_image(img_path, save_path=save_path, show=False)
            cv2.imwrite(str(save_path), result['fused'])
            results.append({
                'file':       str(img_path),
                'pred_class': result['pred_class'],
                'grade_name': result['grade_name'],
                'confidence': round(result['confidence'], 4),
            })
        except Exception as e:
            print(f'Error on {img_path}: {e}')

    print(f'\nProcessed {len(results)}/{min(max_images, len(image_paths))} images.')
    print(f'Saved to: {output_dir}')
    return results


# Uncomment to run batch processing:
# batch_results = batch_process(test_files, OUTPUT_DIR / 'batch', max_images=20)
print('Batch processing function defined. Uncomment to run.')